#### Here we will use a technique called bag of words, it is a matrix having count words w.r.t to a document

####

Then on that resultant matrix we apply SVD
$A=U \sum V^T$


In [3]:
# load a PDF file
import requests
import os

url = 'https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf'

# Get PDF docuement path
file_name = 'human-nutrition-text.pdf'
output_path = f'data/{file_name}'

if not os.path.exists(output_path):
    print(f'[INFO] file doesnt exist, downloading...')
    try:
        response = requests.get(url)

        # check if request was successful
        if response.status_code == 200:
            # open the file and save it
            with open(output_path, 'wb') as f:
                f.write(response.content)

            print(f"[INFO] file has been downloaded and saved as {file_name}");
        else:
            print(f'[INFO] Failed to download the file. Status code: {response.status_code}')

    except Exception as e:
        print(f"An error occurred: {e}")

else:
    print(f'File {output_path} exists')



File data/human-nutrition-text.pdf exists


In [55]:
import fitz #pymupdf
from tqdm.auto import tqdm

# perfoms minor formatting on text
def text_formatter(text: str) -> str:
    cleaned_text = text.replace("\n", " ").strip()

    # potentially more text formatting functions can go here
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = text_formatter(page.get_text())
        if (len(text) > 100):
            pages_and_texts.append({
                "text": text,
                "page_number": len(pages_and_texts) + 1
                }) 
        
    return pages_and_texts


pages_and_texts = open_and_read_pdf(output_path)

1208it [00:01, 641.69it/s]


In [57]:
pages_and_texts[:5]

[{'text': 'Human Nutrition: 2020  Edition  UNIVERSITY OF HAWAI‘I AT MĀNOA  FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM  ALAN TITCHENAL, SKYLAR HARA,  NOEMI ARCEO CAACBAY, WILLIAM  MEINKE-LAU, YA-YUN YANG, MARIE  KAINOA FIALKOWSKI REVILLA,  JENNIFER DRAPER, GEMADY  LANGFELDER, CHERYL GIBBY, CHYNA  NICOLE CHUN, AND ALLISON  CALABRESE',
  'page_number': 1},
 {'text': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program is licensed under a Creative Commons Attribution 4.0  International License, except where otherwise noted.',
  'page_number': 2},
 {'text': 'Contents  Preface  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  xxv  About the Contributors  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  xxvi  Acknowledgements  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  xl  Pa

In [58]:
#build a vocab
vocab = set()

# Loop through num_sentence_chunk_size and split sentences into chunks
for item in tqdm(pages_and_texts):
    topic = item["text"]
    words = topic.strip().split(" ")
    for w in words:
        if (len(w.strip()) > 0):
            vocab.add(w)

100%|██████████| 1147/1147 [00:00<00:00, 19919.78it/s]


In [59]:
vocab = sorted(list(vocab))
word_to_idx = {word: i for i, word in enumerate(vocab)}

In [60]:
len(vocab)

25164

In [80]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [81]:
# bag of words
bow = torch.zeros(len(pages_and_texts), len(vocab), dtype=torch.float, device=device)

In [77]:
def bag_of_words():
    # we could use tokenization, but would use simple technique
    for i, doc in enumerate(pages_and_texts):
        for word in doc["text"].strip().split(" "):
            if word in word_to_idx:
                bow[i, word_to_idx[word]] += 1

bag_of_words()


In [93]:
bow.max(dim=0), bow.shape

(torch.return_types.max(
 values=tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 indices=tensor([0, 0, 0,  ..., 0, 0, 0])),
 torch.Size([1147, 25164]))

In [95]:
#now do SVD on bow
U, S, V = torch.linalg.svd(bow)

In [98]:
U.shape, S.shape, V.shape

(torch.Size([1147, 1147]), torch.Size([1147]), torch.Size([25164, 25164]))

In [ ]:
num_topics = 5 
#taking top 5 axis or directions based on decreasing order of sigular values, unit vectors
# order defines significance of each axis or direction
U = U[:, :5] #five important from C(bow)
S = torch.diag(S[:num_topics])
V = V[:num_topics, :] # five important from C(bow^T) or rowspace of bow

In [109]:
doc_topics = U @ S #scaling variance in the direction of U by amount S
word_topics = V.T @ S # similarly for row space

In [110]:
word_topics.shape, doc_topics.shape

(torch.Size([25164, 5]), torch.Size([5, 5]))

In [115]:
#display top words of each topic
def display_words(num_words = 50):
    print("\n Top words per topic")
    for idx in range(word_topics.size(1)):
        words = word_topics[:, idx]
        top_indices = torch.argsort(words, descending=True)[:num_words]
        top_words = [vocab[i] for i in top_indices]
        print(f"Topic {idx + 1}: {', '.join(top_words)}")


display_words()


 Top words per topic
Topic 1: "Percentage, #2: , #E09/REV., $1, $1,200, $1,429, $10, $134, $174, $20, $470, $68.2, $72, %, %3Arid%3Acrossref.org&rfr_dat=cr_pub%3Dpubmed., %DV, &, (, (%, (%), (.4, (0-12, (0-6, (0.9+1.5=0.1), (0–6, (1, (1), (1-2, (1-3, (1.2, (1.3, (10, (10,000, (10,000+, (100, (1000, (1013)., (10–30, (11, (11,655, (111), (113, (12, (120, (13, (132, (14-18, (143, (14–18, (14–18)
Topic 2: "Percentage, #2: , #E09/REV., $1, $1,200, $1,429, $10, $134, $174, $20, $470, $68.2, $72, %, %3Arid%3Acrossref.org&rfr_dat=cr_pub%3Dpubmed., %DV, &, (, (%, (%), (.4, (0-12, (0-6, (0.9+1.5=0.1), (0–6, (1, (1), (1-2, (1-3, (1.2, (1.3, (10, (10,000, (10,000+, (100, (1000, (1013)., (10–30, (11, (11,655, (111), (113, (12, (120, (13, (132, (14-18, (143, (14–18, (14–18)
Topic 3: "Percentage, #2: , #E09/REV., $1, $1,200, $1,429, $10, $134, $174, $20, $470, $68.2, $72, %, %3Arid%3Acrossref.org&rfr_dat=cr_pub%3Dpubmed., %DV, &, (, (%, (%), (.4, (0-12, (0-6, (0.9+1.5=0.1), (0–6, (1, (1), (1-2, (1-3

In [126]:
#display top topics of each document
def display_topics(num_topics = 5):
    print("\n Top words per topic")
    for idx in range(doc_topics.size(1)):
        topics = doc_topics[:, idx]
        top_indices = torch.argsort(topics, descending=True)[:num_topics]
        top_topics = [pages_and_texts[i]["text"] for i in top_indices]
        print(f"\nDoc {idx + 1}: {'\n\n\t '.join(top_topics)}")
            


display_topics()


 Top words per topic

Doc 1: Human Nutrition: 2020  Edition  UNIVERSITY OF HAWAI‘I AT MĀNOA  FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM  ALAN TITCHENAL, SKYLAR HARA,  NOEMI ARCEO CAACBAY, WILLIAM  MEINKE-LAU, YA-YUN YANG, MARIE  KAINOA FIALKOWSKI REVILLA,  JENNIFER DRAPER, GEMADY  LANGFELDER, CHERYL GIBBY, CHYNA  NICOLE CHUN, AND ALLISON  CALABRESE

	 Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program is licensed under a Creative Commons Attribution 4.0  International License, except where otherwise noted.

	 Contents  Preface  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  xxv  About the Contributors  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  xxvi  Acknowledgements  University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program and Human Nutrition  Program  xl  Part I. Chapter 1. Basic Concepts in Nutritio